# LoRA Fine-Tuning Tutorial

This notebook demonstrates how to fine-tune a pretrained pertTF model on your own dataset using LoRA (Low-Rank Adaptation), and then run inference with the fine-tuned adapter.

# Install necessary packages

In [1]:
!pip install scanpy flash-attn huggingface_hub

# Install pertTF

In [2]:
import sys
# method 1: clone main branch
!git clone https://github.com/davidliwei/pertTF.git
sys.path.insert(0, '/content/pertTF/')

# method 2: pip
#!pip install -i https://test.pypi.org/simple/ pertTF

fatal: destination path 'pertTF' already exists and is not an empty directory.


# 1. Prepare data

pertTF expects an `AnnData` object with:
- **`adata.X`** — log-normalized expression matrix
- **`adata.layers['X_binned']`** — a copy of the expression (used as the training input layer)
- **`adata.obs['celltype']`** — cell type labels
- **`adata.obs['condition']`** (or `'genotype'`) — perturbation labels, with `'ctrl'` for control cells
- **`adata.var['highly_variable']`** — boolean column marking highly-variable genes

In [4]:
from huggingface_hub import hf_hub_download
import scanpy as sc
import anndata as ad
import numpy as np

# Download a demo dataset (pancreatic 18-clone perturbation data)
hf_hub_download(
    repo_id="weililab/pancreatic_18clone",
    filename="./18clones_seurat.h5ad",
    repo_type="dataset",
    local_dir="./",
)
adata = sc.read_h5ad("./18clones_seurat.h5ad")
adata = ad.AnnData(X=adata.raw.X, obs=adata.obs, var=adata.raw.var)

# Create a 'condition' column from the 'gene' column.
# The demo dataset uses labels like "47_WT", "44_HHEX", "46_HHEX_het".
# We strip the numeric prefix to get the gene name, and map WT -> ctrl.
def clean_condition(gene_label):
    if gene_label == "NA" or str(gene_label) == "nan":
        return "ctrl"
    parts = str(gene_label).split("_", 1)
    name = parts[1] if len(parts) > 1 else parts[0]
    if name == "WT":
        return "ctrl"
    # Normalize: HHEX_het -> HHEXhet, FOXA1/2 -> FOXA1
    name = name.replace("_", "").split("/")[0]
    return "ctrl+" + name

adata.obs["condition"] = adata.obs["gene"].map(clean_condition)

# preprocess the data
# log normalized expression data needs to be moved to layer under name 'X_binned'
adata.layers['X_binned'] = adata.X
# find highly variable genes (model expects a highly_variable col in adata.var)
sc.pp.highly_variable_genes(adata, n_top_genes=5000)

print(f"Shape: {adata.shape}")
print(f"Conditions: {adata.obs['condition'].unique().tolist()}")
print(f"Cell types: {adata.obs['celltype'].unique().tolist()}")
print(f"HVGs: {adata.var['highly_variable'].sum()}")

/tmp/ipykernel_986637/837683607.py:33: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["condition"] = adata.obs["gene"].map(clean_condition)


Shape: (3000, 36601)
Conditions: ['ctrl+BCOR', 'ctrl+CCDC6', 'ctrl+HHEX', 'ctrl', 'ctrl+OTUD5', 'ctrl+FOXA1', 'ctrl+HHEXhet', 'ctrl+TADA2B', 'ctrl+UBA6', 'ctrl+KDM2B']
Cell types: ['DE', 'Stromal', 'Liver', 'PP', 'EnP', 'ESC', 'Endothelial']
HVGs: 5000


## 2. Load a pretrained model

We load a pretrained pertTF model from Hugging Face. The `HFPerturbationTFModel` class handles downloading weights, vocabulary, and configuration automatically.

In [5]:
from perttf.model.hf import HFPerturbationTFModel

model = HFPerturbationTFModel.from_pretrained(
    "weililab/pertTF-tiny",
    use_fast_transformer=True,
    fast_transformer_backend="flash",
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Base model loaded: {total_params:,} parameters")

✅ Detected Flash Attention v2.
Model loaded. 100 layers transferred.
Base model loaded: 1,209,567 parameters


## 3. Configure LoRA

`build_lora_config()` returns a PEFT `LoraConfig` with sensible defaults for pertTF. Key parameters:

| Parameter | Description | Typical values |
|-----------|-------------|----------------|
| `r` | Rank of LoRA matrices | 4, 8, 16 |
| `lora_alpha` | Scaling factor (higher = stronger adaptation) | 16, 32 |
| `lora_dropout` | Dropout on LoRA layers | 0.05 - 0.2 |
| `target_modules` | Which layers get LoRA adapters | (defaults below) |

By default, LoRA is applied to: `qkv_proj`, `out_proj`, `linear1`, `linear2`, `decoder.fc.0`, `decoder.fc.2`.

In [6]:
lora_config = model.build_lora_config(
    r=8,              # rank of the low-rank matrices
    lora_alpha=32,    # scaling factor
    lora_dropout=0.1, # dropout on LoRA layers
)
print(lora_config)

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=None, inference_mode=False, r=8, target_modules={'linear1', 'decoder.fc.0', 'out_proj', 'linear2', 'decoder.fc.2', 'qkv_proj'}, lora_alpha=32, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


/home/sam/.local/share/mamba/envs/pertTF/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


## 4. Fine-tune with LoRA

`run_lora_train()` handles the full training pipeline:
1. Wraps the base model with PEFT/LoRA (only adapter weights are trained)
2. Creates train/validation data loaders from your AnnData
3. Trains with best-model checkpointing based on validation MSE
4. Restores the best checkpoint and saves the adapter to disk

In [7]:
SAVE_DIR = "./my_lora_adapter"

peft_model = model.run_lora_train(
    adata=adata,
    epochs=1,
    batch_size=8,
    lr=1e-3,
    train_val_split=0.2,       # 80/20 train/validation split
    lora_config=lora_config,
    save_dir=SAVE_DIR,
    seed=42,
)

# Training prints validation MSE after each epoch and automatically
# restores the best checkpoint (lowest validation MSE) at the end.

[run_lora_train] filtered ps columns to existing obs columns: None
[run_lora_train] model genotype_to_index missing labels from current adata; rebuilding genotype mapping from adata.
Initializing PertTFUniDataManager: Creating vocab and mappings...
sampling_mode is hvg, sampling 2000 HVGs + 1000 non-HVGs for training
Initialization complete.
Creating a single train/validation split (test_size=0.2)...


/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/utils/set_optimizer.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=config.amp)
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` ins

pertTF - INFO - | epoch   1 | 200/300 batches | lr 0.00100000 | ms/batch 16.83 | loss 37.11 | mse  0.10 | mre 68493.93 |mse_next  0.10 | mre_next 68493.93 |cls  1.64 | pert  2.90 | ps  0.00 | ps_next  0.00 |gepc  0.08 |gepc_next  1.19 |


/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Epoch 1/1 | val_mse: 0.0863
Restored best model from epoch 1 (val_mse=0.0863)
Saved PEFT adapter to: /home/sam/dev/Genomics/pertTF/demos/tutorials/my_lora_adapter


In [8]:
# Check trainable parameter counts
total = sum(p.numel() for p in peft_model.parameters())
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"Total parameters:     {total:,}")
print(f"Trainable (LoRA):     {trainable:,}")
print(f"Trainable fraction:   {100.0 * trainable / total:.2f}%")

Total parameters:     1,216,487
Trainable (LoRA):     6,920
Trainable fraction:   0.57%


In [9]:
# Check what was saved
import os

for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:40s} {size / 1024:.1f} KB")

  README.md                                4.9 KB
  adapter_config.json                      0.8 KB
  adapter_model.safetensors                30.0 KB


## 5. Inference with LoRA adapters

Below we demonstrate two inference tasks, each using a LoRA-adapted model. Each task uses a different pretrained base model — you fine-tune each one separately and use the resulting adapter.

> **Note:** A LoRA adapter is specific to the base model it was trained on. You cannot mix adapters across different base models.

In [10]:
from peft import PeftModel
from perttf.model.hf import HFPerturbationTFModel
from perttf.model.train_function import eval_testdata
import numpy as np

# Define a reusable evaluation wrapper that works with both base and PEFT models
def eval_wrapper(model, adata_test, expression=False):
    bm = model.get_base_model() if hasattr(model, "get_base_model") else model
    res = eval_testdata(
        model,
        adata_test,
        None,
        train_data_dict={
            "genotype_to_index": bm.genotype_to_index,
            "vocab": bm.vocab,
            "cell_type_to_index": bm.cell_type_to_index,
        },
        config=bm.training_config,
        mvc_full_expr=expression,
        predict_expr=expression,
    )
    return res

### 5a. Classification task

Fine-tune `pertTF-tiny` with LoRA, then predict genotype and cell type labels.

In [11]:
# 1. Load the same base model used for fine-tuning
base_model = HFPerturbationTFModel.from_pretrained(
    "weililab/pertTF-tiny",
    use_fast_transformer=True,
    fast_transformer_backend="flash",
)
base_model.to("cuda")

# 2. Apply the saved LoRA adapter
peft_model = PeftModel.from_pretrained(base_model, SAVE_DIR)
peft_model.eval()

# 3. Run inference — returns predicted genotype and cell type
adata_eva = eval_wrapper(peft_model, adata)
print("Predicted genotypes:")
print(adata_eva.obs["predicted_genotype"].value_counts().head())
print("\nPredicted cell types:")
print(adata_eva.obs["predicted_celltype"].value_counts().head())

Model loaded. 100 layers transferred.
pertTF - INFO - 36601 genes shared between model vocab's 36604 and anndata's 36601 genes
pertTF - INFO - Evaluating data using 6000 tokens for each cell


100%|███████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:01<00:00, 12.41it/s]
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:803: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata_t.obsm["X_scGPT"] = cell_embeddings


Predicted genotypes:
predicted_genotype
TADA2B    2865
OTUD5      131
WT           3
ARX          1
Name: count, dtype: int64

Predicted cell types:
predicted_celltype
DE             1407
Liver           871
PFG             515
Stromal         158
Endothelial      45
Name: count, dtype: int64


### 5b. Perturbation prediction (expression)

Fine-tune `pertTF-perturb_5k_mvc_only` with LoRA, then predict gene expression after applying a perturbation. This model uses a 5K-gene vocabulary, so we subset the data to matching genes first.

In [12]:
PERTURB_ADAPTER_DIR = "./lora_perturb_adapter"

perturb_model = HFPerturbationTFModel.from_pretrained(
    "weililab/pertTF-perturb_5k_mvc_only",
    use_fast_transformer=True,
    fast_transformer_backend="flash",
)

# Subset adata to genes in the model's vocabulary
shared_genes = adata.var.index.isin(perturb_model.vocab.stoi)
adata_pert = adata[:, shared_genes].copy()
adata_pert.layers["X_binned"] = adata_pert.X

# the perturb 5k model works on 5K HVGs that were in the training data, thus we want to use them all
# evaluation will subset your adata to the 5000 HVGs before inference
if perturb_model.training_config["sampling_mode"] == "hvg":
    adata_pert.var["highly_variable"] = True

# to initiate perturbations set target perturbations using the genotype_next column
# assuming all cells in adata are non-perturbed (perturbed randomly for demo)
adata_pert.obs["genotype_next"] = np.random.choice(["FOXA2", "PDX1"], adata_pert.shape[0])

# Fine-tune with LoRA
lora_config = perturb_model.build_lora_config(r=8, lora_alpha=32, lora_dropout=0.1)
peft_perturb = perturb_model.run_lora_train(
    adata=adata_pert,
    epochs=5,
    batch_size=8,
    lr=1e-3,
    lora_config=lora_config,
    save_dir=PERTURB_ADAPTER_DIR,
)

# to initiate perturbations set target perturbations using the genotype_next column
# assuming all cells in adata are non-perturbed (perturbed randomly for demo)
adata_pert.obs["genotype_next"] = np.random.choice(["FOXA2", "PDX1"], adata_pert.shape[0])

# Run with expression=True to get predicted expression values
adata_eva = eval_wrapper(peft_perturb, adata_pert, expression=True)

# corresponding perturbations are found in:
print("Target perturbations:", adata_eva.obs["genotype_next"].value_counts().to_dict())
# perturbed expressions are found:
print("Perturbed expression shape:", adata_eva.obsm["mvc_next_expr"].shape)

Model loaded. 100 layers transferred.
[run_lora_train] filtered ps columns to existing obs columns: None
[run_lora_train] model genotype_to_index missing labels from current adata; rebuilding genotype mapping from adata.
Initializing PertTFUniDataManager: Creating vocab and mappings...
sampling_mode is hvg, sampling 3000 HVGs + 1000 non-HVGs for training
Initialization complete.
Creating a single train/validation split (test_size=0.2)...


/tmp/ipykernel_986637/332577800.py:18: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata_pert.var["highly_variable"] = True
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/utils/set_optimizer.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=config.amp)
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics

Epoch 1/5 | val_mse: 0.1931


/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:487: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):
/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Epoch 2/5 | val_mse: 0.1921


...

/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:487: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=config.amp):


pertTF - INFO - Evaluating data using 5055 tokens for each cell


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 11.82it/s]

Target perturbations: {'PDX1': 510, 'FOXA2': 490}
Perturbed expression shape: (1000, 5054)



/home/sam/dev/Genomics/pertTF/demos/tutorials/../../perttf/model/train_function.py:803: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata_t.obsm["X_scGPT"] = cell_embeddings


## 6. Additional training options

Here are some useful options for tuning the training process:

In [ ]:
# For larger datasets or GPU memory constraints, enable mixed precision:
#
# peft_model = model.run_lora_train(
#     adata=adata,
#     epochs=10,
#     batch_size=16,
#     lr=5e-4,
#     lora_config=lora_config,
#     save_dir="./my_lora_adapter",
#     amp=True,              # enable automatic mixed precision
#     amp_dtype="bf16",      # use bfloat16 (or "fp16")
#     log_interval=100,      # print training loss every N batches
#     seed=42,
# )
#
# You can also override which layers get LoRA adapters:
#
# lora_config = model.build_lora_config(
#     r=16,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     target_modules=["qkv_proj", "out_proj"],  # only attention layers
# )